# 5-2 Quantization 초보자용 실습

이 실습은 **모델 양자화가 무엇인지 처음 접하는 사람**을 위한 아주 쉬운 버전입니다.

오늘 할 일은 딱 3가지입니다.

1. FP16 모델을 불러온다.
2. INT4 양자화 모델을 불러온다.
3. **메모리 사용량**과 **응답 결과**를 비교한다.

## 이 실습에서 꼭 기억할 것

- **양자화(Quantization)** = 숫자를 더 적은 비트로 저장하는 것
- 비트 수가 줄어들면 **메모리를 덜 사용**한다
- 대신 아주 약간의 **정확도 손실**이 생길 수 있다

- FP(Floating Point): 소수점을 표현할 수 있는 부동소수점 형식
- INT(Integer): 정수만 표현하는 형식 (더 단순하여 비트를 절약 가능)

예시:
- FP32: 32비트 부동소수점 (가장 정밀)
- FP16: 16비트 부동소수점
- INT4:  4비트 정수 (가장 가볍지만 표현 범위 제한)

## 실습 목표

실습이 끝나면 아래를 설명할 수 있으면 됩니다.

- FP16과 INT4의 차이
- 왜 양자화를 하면 메모리가 줄어드는지
- 양자화 후에도 모델이 답변은 가능한지

## 1. 라이브러리 설치

처음 한 번만 실행하면 됩니다.

In [16]:
%pip install -q "transformers>=4.55.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. 기본 라이브러리 불러오기

In [17]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## 3. 실행 환경 확인

- GPU가 있으면 실습이 훨씬 편합니다.
- bitsandbytes 4비트 양자화는 보통 **CUDA GPU 환경**에서 사용합니다.

In [18]:
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print(f"GPU 메모리: {gpu.total_memory / 1024**3:.2f} GB")
else:
    print("GPU가 없으면 INT4 실습이 제대로 동작하지 않을 수 있습니다.")

torch version: 2.10.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
GPU 메모리: 15.93 GB


## 4. 사용할 모델 이름 정하기

너무 큰 모델보다 **실습용으로 비교적 가벼운 모델**을 사용합니다.

In [19]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print("사용 모델:", model_name)

사용 모델: Qwen/Qwen2.5-1.5B-Instruct


## 5. 메모리 측정용 함수 만들기

아래 함수는 GPU 메모리를 보기 쉽게 GB 단위로 바꿔줍니다.

In [20]:
def gpu_memory_gb():
    if not torch.cuda.is_available():
        return 0.0
    # reserved = PyTorch가 OS에서 확보한 전체 GPU 메모리 (모델 비교에 적합)
    return torch.cuda.memory_reserved() / 1024**3


def reset_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

## 6. FP16 모델 불러오기

여기서는 양자화를 하지 않은 기본 비교 대상을 만듭니다.

In [21]:
reset_gpu_memory()

print("FP16 모델 로딩 중...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

fp16_model_memory = gpu_memory_gb()
print(f"FP16 모델 로딩 후 메모리: {fp16_model_memory:.2f} GB")

FP16 모델 로딩 중...
FP16 모델 로딩 후 메모리: 4.11 GB


## 7. 간단한 추론 함수 만들기

이 함수는
- 질문을 넣고
- 답변을 받고
- 걸린 시간을 측정합니다.

In [22]:
def ask_model(model, tokenizer, prompt, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start

    answer = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return answer, elapsed

## 8. FP16 모델에게 질문해 보기

In [23]:
prompt = "서울에서 부산까지 KTX로 보통 얼마나 걸리나요?"

fp16_answer, fp16_time = ask_model(model_fp16, tokenizer, prompt)
print("[FP16 답변]")
print(fp16_answer)
print(f"FP16 추론 시간: {fp16_time:.2f}초")

[FP16 답변]
서울에서 부산까지의 KTX(고속철도)는 일반적으로 약 3시간 20분 정도 소요됩니다. 하지만 실제 시간은 운행 환경과 기상条件에 따라 달라질 수 있으니, 예약 시 참고해 주세요. 또한, 여름철에는 더 빠른 시간으로 도착할 수도
FP16 추론 시간: 2.44초


## 9. FP16 모델의 대략적인 파라미터 수 확인

정확한 내부 구조를 몰라도, 파라미터 수가 많을수록 메모리가 많이 필요하다는 점만 이해하면 충분합니다.

In [24]:
total_params = sum(p.numel() for p in model_fp16.parameters())
print(f"총 파라미터 수: {total_params:,}")

print("이론상 필요한 메모리(가중치만 단순 계산)")
print(f"FP32: {total_params * 4 / 1024**3:.2f} GB")
print(f"FP16: {total_params * 2 / 1024**3:.2f} GB")
print(f"INT4 : {total_params * 4 / 8 / 1024**3:.2f} GB  ← 4bit/8 = 0.5 bytes/param")
print("  * 실제 메모리는 스케일 팩터(FP16) 오버헤드로 인해 더 클 수 있음")

총 파라미터 수: 1,543,714,304
이론상 필요한 메모리(가중치만 단순 계산)
FP32: 5.75 GB
FP16: 2.88 GB
INT4 : 0.72 GB  ← 4bit/8 = 0.5 bytes/param
  * 실제 메모리는 스케일 팩터(FP16) 오버헤드로 인해 더 클 수 있음


## 10. FP16 모델 정리

이제 같은 모델을 INT4로 다시 불러오기 위해 메모리를 비웁니다.

In [25]:
import gc
del model_fp16
gc.collect()
reset_gpu_memory()
reset_gpu_memory()
print("FP16 모델 메모리 정리 완료")

FP16 모델 메모리 정리 완료


## 11. INT4 양자화 설정 만들기

핵심은 `load_in_4bit=True` 입니다.

- 모델 가중치를 4비트로 저장
- 연산은 보통 FP16으로 진행

In [26]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(quant_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## 12. INT4 모델 불러오기

In [27]:
if not torch.cuda.is_available():
    raise RuntimeError("이 실습의 INT4 부분은 CUDA GPU 환경을 권장합니다.")

reset_gpu_memory()

print("INT4 모델 로딩 중...")
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

int4_model_memory = gpu_memory_gb()
print(f"INT4 모델 로딩 후 메모리: {int4_model_memory:.2f} GB")

INT4 모델 로딩 중...
INT4 모델 로딩 후 메모리: 2.94 GB


## 13. INT4 모델에게 같은 질문해 보기

질문을 똑같이 넣어서 비교해야 차이가 잘 보입니다.

In [28]:
int4_answer, int4_time = ask_model(model_int4, tokenizer, prompt)
print("[INT4 답변]")
print(int4_answer)
print(f"INT4 추론 시간: {int4_time:.2f}초")

[INT4 답변]
서울과 부산을 연결하는 KTX(高速鉄路)는 일반적으로 2시간 30분 정도가 걸립니다. 하지만 실제 시간은 운행 빈도와 상황에 따라 달라질 수 있습니다. 예를 들어, 대기 시간이 길거나 교통상황이 좋지 않다면 더 오래걸릴 수도
INT4 추론 시간: 3.65초


## 14. FP16과 INT4 결과 비교

여기서 가장 중요한 것은 **메모리가 줄었는지**입니다.

In [29]:
memory_saved = fp16_model_memory - int4_model_memory
memory_saved_ratio = (memory_saved / fp16_model_memory * 100) if fp16_model_memory > 0 else 0

print("=" * 60)
print("비교 결과")
print("=" * 60)
print(f"FP16 모델 메모리 : {fp16_model_memory:.2f} GB")
print(f"INT4 모델 메모리 : {int4_model_memory:.2f} GB")
print(f"메모리 절감량     : {memory_saved:.2f} GB")
print(f"메모리 절감 비율 : {memory_saved_ratio:.1f}%")
print()
print(f"FP16 추론 시간   : {fp16_time:.2f}초")
print(f"INT4 추론 시간   : {int4_time:.2f}초")

비교 결과
FP16 모델 메모리 : 4.11 GB
INT4 모델 메모리 : 2.94 GB
메모리 절감량     : 1.17 GB
메모리 절감 비율 : 28.4%

FP16 추론 시간   : 2.44초
INT4 추론 시간   : 3.65초


## 15. 결과 해석

아래 질문에 스스로 답해보세요.

1. INT4가 FP16보다 메모리를 덜 썼는가?
2. 답변 품질은 완전히 망가졌는가, 아니면 어느 정도 유지되는가?
3. 실무에서 **조금의 품질 손해와 큰 메모리 절약** 중 무엇이 더 중요한 상황이 있을까?

## 16. 한 줄 정리

- **양자화는 모델을 더 가볍게 만드는 방법**입니다.
- FP16보다 INT4가 보통 메모리를 훨씬 덜 사용합니다.
- 대신 경우에 따라 답변 품질이 조금 떨어질 수 있습니다.

## 초보자용 과제

### 과제 1
질문을 아래처럼 바꿔서 다시 실행해보세요.
- "대한민국의 수도는 어디인가요?"
- "파이썬이 무엇인가요?"
- "자기소개를 2문장으로 해주세요."

### 과제 2
직접 한 문장으로 정리해보세요.
- "양자화란 __________ 이다."